setfit prefers transformers 4.48 

In [21]:
import torch
print(torch.__version__)
print(torch.__file__)

USE_XPU = False

if USE_XPU and torch.xpu.is_available():
    device = torch.device("xpu:0")
    torch.set_default_dtype(torch.bfloat16)
    available, total = torch.xpu.mem_get_info(device)
    print(f"Using {device}: {total / 1024 ** 3:.1f} GB, {100 * available / total:.1f}% available.")
elif torch.cuda.is_available():
    device = torch.device("cuda:0")
    torch.set_default_dtype(torch.float32)
    print(f"Using {device}")
else:
    device = torch.device("cpu")
    torch.set_default_dtype(torch.float32)
    print(f"Using {device}")

2.8.0+xpu
/home/urdatorn/.pyenv/versions/xpu/lib/python3.13/site-packages/torch/__init__.py
11.3 GB, 0.4% tillgängligt.


In [ ]:
import transformers.training_args as training_args
from transformers.integrations.integration_utils import default_logdir


training_args.default_logdir = default_logdir

In [ ]:
from sentence_transformers import SentenceTransformer
from setfit import SetFitModel

body = SentenceTransformer("sentence-transformers/all-mpnet-base-v2", device=str(device))
model = SetFitModel(model_body=body)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 19246.58it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [15]:
from datasets import load_dataset

dataset = load_dataset("DigPhil/sentiment-grc")

train_dataset = dataset["train"]
val_dataset = dataset["validation"]
test_dataset = dataset["test"]

In [16]:
from setfit import TrainingArguments

args = TrainingArguments(
    batch_size=32,
    num_epochs=10,
)

In [17]:
from setfit import Trainer

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
)

Map: 100%|██████████| 578/578 [00:00<00:00, 42296.49 examples/s]


In [22]:
trainer.train()

***** Running training *****
  Num unique pairs = 218250
  Batch size = 32
  Num epochs = 10


RuntimeError: Native API failed. Native API returns: 20 (UR_RESULT_ERROR_DEVICE_LOST)